# Этап №3. | AI/ML | Парсинг данных

**Версия 4.** Скриншоты заменены ячейками с кодом: отчёт появляется в выводе ячейки.

**Цель этапа:** подготовка нормативной документации (ПУЭ) для векторного поиска: от исходников до загрузки в векторную базу.

**Ссылка на проект:** [https://github.com/kirag-ozyaz/Diplom_AI/](https://github.com/kirag-ozyaz/Diplom_AI/)

**Структура проекта:**
```text
energy_norms_bot/
├── infra/                              # Инфраструктура и развертывание
│   ├── docker/                         # Docker-конфигурации
│   │   └── Dockerfile.attu/            # Docker для Attu UI (Milvus Web UI)
│   └── milvus/                         # Конфигурация Milvus векторной БД
├── src/                                # Исходный код приложения
│   └── preprocessing/                  # Модули предобработки документов (3 Этап)
│       ├── Create_mds/                 # Шаг 1: DOCX → Markdown
│       ├── Create_chunkeds/            # Шаг 2: Markdown → Chunks
│       └── Create_embeddings/          # Шаг 3: Chunks → Векторная БД
└── data/                               # Данные проекта
    ├── raw/                            # Исходные документы (PDF/DOCX)
    │   ├── База исходники/             # Исходные файлы документов
    │   └── Нормативная база/           # Нормативные документы
    ├── extracted/                      # Извлеченный текст в Markdown (*.md + image_*/)
    └── chunked/                        # Сегментированные данные (*.jsonl + image_*/)
```

**Пайплайн подготовки (парсинга) данных:** `raw` → `extracted` → `chunked` → `embeddings` (векторная база)


**Содержание:**
1. [Конвертация исходных файлов raw](#sec-1)
2. [Редактирование файлов extracted](#sec-2)
3. [Чанкирование файлов extracted](#sec-3)
4. [Векторная база данных](#sec-4)
5. [Загрузка в векторную базу и тестовые запросы](#sec-5)


**Как читать отчёт.** Скриншоты заменены ячейками с кодом. Сначала выполните ячейку подготовки: она находит корень проекта. Остальные ячейки печатают отчёты по файлам и, если запущен Milvus, по коллекции `diplom_multimodal`.

Повторная загрузка в векторную базу из этого ноутбука не запускается: в конфиге у `load_data` стоит `drop_existing: true`, и коллекция была бы пересоздана.


In [1]:
from pathlib import Path
import json
import socket


def find_root() -> Path:
    """Корень репозитория: каталог, где лежат data/extracted и src/preprocessing."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "extracted").is_dir() and (candidate / "src" / "preprocessing").is_dir():
            return candidate
    raise FileNotFoundError(
        "Корень проекта не найден. Запустите ноутбук из каталога репозитория "
        "или из Этапы/Reports."
    )


def milvus_settings():
    cfg = json.loads((ROOT / "config" / "rag_runtime.json").read_text(encoding="utf-8"))
    vector_db = cfg["vector_db"]
    return vector_db["host"], str(vector_db["port"]), vector_db["collection_name"]


def milvus_reachable(host: str, port: str, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except OSError:
        return False


def connect_milvus():
    """Подключение к уже загруженной коллекции. None, если сервер или пакет недоступны."""
    host, port, name = milvus_settings()
    print(f"Сервер: {host}:{port}")
    print(f"Коллекция: {name}")
    if not milvus_reachable(host, port):
        print("Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.")
        return None
    try:
        from pymilvus import Collection, connections, utility
    except ImportError:
        print("Пакет pymilvus не установлен. Отчёт по коллекции появится после установки pymilvus.")
        return None
    alias = "readme"
    if connections.has_connection(alias):
        connections.disconnect(alias)
    connections.connect(alias=alias, host=host, port=port)
    print(f"Подключение установлено: {host}:{port}")
    return {"alias": alias, "name": name, "utility": utility, "Collection": Collection}


ROOT = find_root()
print(f"Корень проекта: {ROOT}")


Корень проекта: X:\Учеба_УИИ\Итоговы_Проект


## 1. Конвертация исходных файлов raw<a id="sec-1"></a>


Выполнена конвертация исходных документов MS Word (формат DOCX) нормативной документации «ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК» (ПУЭ) в формат Markdown (MD).

**Цель конвертации:** Преобразование структурированных документов в текстовый формат, удобный для дальнейшей обработки и построения векторной базы знаний.

**Входные данные:** исходные файлы формата DOCX из директории `data/raw`.

**Основные результаты:**
- Сконвертированные файлы сохранены в директории [data/extracted](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/data/extracted)
- Изображения из документов извлечены и размещены в отдельных папках с префиксом `image_*`
- Сохранена структура документа с заголовками, списками и ссылками на изображения
- Каждая глава ПУЭ представлена отдельным MD-файлом (например: `1.1.md`, `1.2.md`, ..., `7.10.md`)

**Процесс конвертации:**
1. Извлечение изображений из DOCX-файлов.
2. Преобразование форматирования документа в Markdown-разметку.
3. Создание ссылок на извлеченные изображения.
4. Сохранение результатов в структурированном виде.

Подробное описание каждого подшага представлено в следующих разделах.


### 1.1. Исходные данные (raw)


Исходные данные для конвертации представляют собой нормативную документацию в формате Microsoft Word (DOCX).

**Расположение исходных файлов:**

| Путь | Содержимое |
|------|------------|
| [data/raw/Нормативная база/ПУЭ/](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/data/raw/Нормативная%20база/ПУЭ/) | Исходные DOCX-файлы ПУЭ (Правила устройства электроустановок) |
| [data/raw/База исходники/](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/data/raw/База%20исходники/ПУЭ/) | Прочие нормативные документы в формате DOCX |

**Характеристики исходных данных:**

- Формат документов: Microsoft Word (DOCX)
- Структура: каждая глава ПУЭ находится в отдельном файле
- Содержимое: текст, таблицы, изображения, форматирование
- Общее количество файлов: более 70 глав ПУЭ

Структура каталогов с исходными файлами выводится следующей ячейкой:



In [2]:
print("Команда: tree data/raw")
print()


def print_tree(path: Path, prefix: str = "", depth: int = 0, max_depth: int = 4) -> None:
    entries = sorted(path.iterdir(), key=lambda item: (item.is_file(), item.name.lower()))
    dirs = [item for item in entries if item.is_dir()]
    files = [item for item in entries if item.is_file()]
    shown = list(dirs)
    file_note = None
    if files and depth < max_depth:
        if len(files) <= 8:
            shown.extend(files)
        else:
            file_note = f"[{len(files)} файлов: {', '.join(item.name for item in files[:3])}, ...]"
    for index, item in enumerate(shown):
        last = index == len(shown) - 1 and file_note is None
        branch = "└── " if last else "├── "
        print(prefix + branch + item.name)
        if item.is_dir() and depth + 1 <= max_depth:
            next_prefix = prefix + ("    " if last else "│   ")
            print_tree(item, next_prefix, depth + 1, max_depth)
    if file_note:
        branch = "└── "
        print(prefix + branch + file_note)


raw = ROOT / "data" / "raw"
print(raw)
print_tree(raw)

print()
print("Команда: Get-ChildItem data/raw")
print(f"{'Режим':<8} {'Имя'}")
for item in sorted(raw.iterdir(), key=lambda entry: entry.name.lower()):
    mode = "d-----" if item.is_dir() else "a-----"
    print(f"{mode:<8} {item.name}")


Команда: tree data/raw

X:\Учеба_УИИ\Итоговы_Проект\data\raw
├── База исходники
│   ├── всё ПУЭ WORD
│   │   └── всё ПУЭ WORD.7z
│   └── Приказ Минэнерго РФ от 12.08.2022 N 811 Об утверждении Правил технической эксплуатации.rtf
└── Нормативная база
    ├── ПТЭ
    │   ├── DOCX
    │   │   └── Приказ Минэнерго РФ от 12.08.2022 N 811 Об утверждении Правил технической эксплуатации.docx
    │   └── Приказ Минэнерго РФ от 12.08.2022 N 811 Об утверждении Правил технической эксплуатации.md.7z
    └── ПУЭ
        ├── DOCX
        │   └── [44 файлов: 1.1.docx, 1.2.docx, 1.3.docx, ...]
        ├── link.md
        └── ПУЭ_md.7z

Команда: Get-ChildItem data/raw
Режим    Имя
d-----   База исходники
d-----   Нормативная база


### 1.2. Результат конвертации (extracted)


После выполнения конвертации получены следующие результаты:

**Структура выходных данных:**

- Директория: [data/extracted](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/data/extracted)
- Файлы формата Markdown (*.md) по главам: 1.1.md, 1.2.md, ..., 7.10.md
- Рядом с каждым .md-файлом создана папка с изображениями соответствующей главы
- Именование папок с изображениями: `image_[номер_главы]` (например, `image_1.7`)

**Особенности конвертированных файлов:**

- Сохранена иерархическая структура документа
- Заголовки преобразованы в Markdown-разметку (###, ##, #)
- Изображения вынесены в отдельные папки со ссылками в тексте
- Таблицы сохранены в Markdown-формате

**Пример результата:**

Следующие две ячейки печатают фрагмент `data/extracted/1.7.md`: первые 30 строк и участок со ссылками на рисунки 1.7.3 и 1.7.4.



In [3]:
path = ROOT / "data" / "extracted" / "1.7.md"
print(f"Файл: {path}")
print("Первые 30 строк")
print()
lines = path.read_text(encoding="utf-8").splitlines()
for number, line in enumerate(lines[:30], start=1):
    print(f"{number:4}| {line}")


Файл: X:\Учеба_УИИ\Итоговы_Проект\data\extracted\1.7.md
Первые 30 строк

   1| ### ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК
   2| ## Раздел 1 ОБЩИЕ ПРАВИЛА
   3| # Глава 1.7 ЗАЗЕМЛЕНИЕ И ЗАЩИТНЫЕ МЕРЫ ЭЛЕКТРОБЕЗОПАСНОСТИ
   4| 
   5| # Предисловие
   6| 
   7| РАЗРАБОТАНО с учетом требований государственных стандартов, строительных норм и правил, рекомендаций научно-технических советов по рассмотрению проектов глав. Проекты глав рассмотрены рабочими группами Координационного совета по пересмотру ПУЭ.
   8| 
   9| ПОДГОТОВЛЕНО ОАО "НИИпроектэлектромонтаж" совместно с Ассоциацией "Росэлектромонтаж", а подглава "Передвижные электроустановки" - при участии ЦНИИИ N 15 МО РФ, подглава "Электроустановки помещений для содержания животных" - при участии ВИЭСХ.
  10| 
  11| СОГЛАСОВАНО в установленном порядке с Госстроем России, Госгортехнадзором России, РАО "ЕЭС России" (ОАО "ВНИИЭ") и представлено к утверждению Госэнергонадзором Минэнерго России.
  12| 
  13| УТВЕРЖДЕНО Министерством энергетики Рос

In [4]:
path = ROOT / "data" / "extracted" / "1.7.md"
lines = path.read_text(encoding="utf-8").splitlines()
start = next(i for i, line in enumerate(lines) if "Рис.1.7.3" in line)
end = next(i for i, line in enumerate(lines) if "Рис.1.7.4" in line)
start = max(0, start - 12)
end = min(len(lines), end + 3)
print(f"Файл: {path}")
print(f"Строки {start + 1}–{end}: рисунки 1.7.3 и 1.7.4")
print()
for number, line in enumerate(lines[start:end], start=start + 1):
    print(f"{number:4}| {line}")


Файл: X:\Учеба_УИИ\Итоговы_Проект\data\extracted\1.7.md
Строки 52–77: рисунки 1.7.3 и 1.7.4

  52| $б$
  53| 
  54| Рис.1.7.2. Система $TN-S$ переменного ($a$) и постоянного ($b$) тока.
  55| 
  56| Нулевой защитный и нулевой рабочий проводники разделены:  
  57| *1* - заземлитель нейтрали источника переменного тока; *1-1*- заземлитель вывода источника постоянного тока; *1-2* - заземлитель средней точки источника постоянного тока; *2* - открытые проводящие части; *3* - источник питания система $TN-C-S$ - система $TN$, в которой функции нулевого защитного и нулевого рабочего проводников совмещены в одном проводнике в какой-то ее части, начиная от источника питания (рис.1.7.3);
  58| 
  59| ![image9_а.png](image_1.7/image9_а.png)  
  60| $а$  
  61| ![image9_б.png](image_1.7/image9_б.png)  
  62| $б$
  63| 
  64| Рис.1.7.3. Система $TN-C-S$ переменного ($а$) и постоянного ($б$) тока.
  65| 
  66| Нулевой защитный и нулевой рабочий проводники совмещены в одном проводнике в части системы: 

### 1.3. Используемые скрипты


Для автоматизации процесса конвертации были разработаны специализированные Python-скрипты.

Директория: [src/preprocessing/Create_mds](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/src/preprocessing/Create_mds)

**Основные компоненты:**

| Скрипт | Назначение | Основные функции |
|--------|------------|------------------|
| [docx_to_md_images_1.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_mds/docx_to_md_images_1.py) | Обработка одного файла | Извлечение изображений, конвертация форматирования, создание Markdown-файла |
| [generator.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_mds/generator.py) | Массовая обработка | Пакетная обработка всех файлов в директории, логирование процесса |

**Процесс работы скриптов:**

1. **docx_to_md_images_1.py** выполняет следующие операции:
   - Открывает DOCX-файл и анализирует его структуру
   - Извлекает все изображения и сохраняет их в отдельную папку
   - Преобразует форматирование текста в Markdown-разметку
   - Создает ссылки на извлеченные изображения
   - Сохраняет результат в MD-файл

2. **generator.py** обеспечивает:
   - Автоматический обход всех DOCX-файлов в исходной директории
   - Вызов функции конвертации для каждого файла
   - Логирование процесса обработки
   - Обработку ошибок и создание отчета о результатах

**Пример работы:**

Следующая ячейка запускает конвертацию `7.5.docx` во временный каталог и печатает лог: извлечение изображений, папка `image_7.5` и размер Markdown. Файл в `data/extracted` при этом не перезаписывается.



In [ ]:
import sys
import tempfile
from pathlib import Path

docx = ROOT / "data" / "raw" / "Нормативная база" / "ПУЭ" / "DOCX" / "7.5.docx"
script = ROOT / "src" / "preprocessing" / "Create_mds" / "docx_to_md_images_1.py"
print("Команда:")
print(f'python "{script}" "{docx}" <временный_каталог>')
print("Временный каталог нужен, чтобы не перезаписать data/extracted.")
print()

saved_md = ROOT / "data" / "extracted" / "7.5.md"
saved_images = ROOT / "data" / "extracted" / "image_7.5"


def print_saved_result() -> None:
    print("Уже сохранённый результат конвертации:")
    print(f"  Markdown: {saved_md} ({'есть' if saved_md.exists() else 'нет'})")
    if saved_images.is_dir():
        images = sorted(p.name for p in saved_images.iterdir() if p.is_file())
        print(f"  Изображения: {saved_images}")
        print(f"  Найдено изображений: {len(images)}")
        for name in images[:8]:
            print(f"    {name}")
        if len(images) > 8:
            print(f"    ... ещё {len(images) - 8}")
    else:
        print(f"  Папка изображений не найдена: {saved_images}")


if not docx.exists():
    print(f"Исходный файл не найден: {docx}")
    print_saved_result()
else:
    sys.path.insert(0, str(script.parent))
    try:
        from docx_to_md_images_1 import docx_to_md_with_images
    except ImportError as error:
        print(f"Повторный запуск конвертации пропущен: {error}")
        print_saved_result()
    else:
        with tempfile.TemporaryDirectory(prefix="pue_7_5_") as temp_dir:
            markdown = docx_to_md_with_images(str(docx), temp_dir)
            image_dir = Path(temp_dir) / "image_7.5"
            images = list(image_dir.glob("*")) if image_dir.is_dir() else []
            print(f"📄 Markdown (временный, не записан в data/extracted): {len(markdown)} символов")
            print(f"🖼️  Изображений извлечено: {len(images)}")


## 2. Редактирование файлов extracted<a id="sec-2"></a>


Проведена постобработка сконвертированных файлов Markdown (extracted) нормативной документации ПУЭ для обеспечения качества данных при дальнейшем чанкировании и использовании в RAG-системе.

**Цель редактирования:** Подготовка MD-файлов к корректной индексации и векторному поиску: устранение артефактов конвертации и улучшение структуры контента для эффективного извлечения релевантной информации.

**Основные результаты:**
- Отредактированы файлы `extracted/*.md` (глава 1) для отработки пайплайна.
- Устранены дублирующиеся изображения в тексте.
- Формулы и таблицы выделены в отдельные блоки для корректного отображения в RAG.
- Сохранена семантическая структура документов.

**Процесс редактирования:**
1. Анализ сконвертированных MD-файлов на наличие артефактов.
2. Удаление повторяющихся изображений.
3. Выделение формул (LaTeX) и таблиц в отдельные блоки.
4. Проверка целостности структуры документа.

Подробное описание с примерами представлено в следующих подразделах.



### 2.1. Пример правок (пункт 2.5.11)


Ниже представлен пример редактирования фрагмента из главы 2.5 — пункт **2.5.11. Основными характеристиками нагрузок являются их нормативные значения, которые устанавливаются настоящими Правилами, а для нагрузок, не регламентированных ими, - в соответствии со строительными нормами и правилами**.

Следующая ячейка печатает пункт 2.5.11 из отредактированного `data/extracted/2.5.md`: формулы записаны в LaTeX, повторные ссылки на одно изображение убраны.


In [5]:
path = ROOT / "data" / "extracted" / "2.5.md"
lines = path.read_text(encoding="utf-8").splitlines()
start = next(i for i, line in enumerate(lines) if line.startswith("2.5.11."))
end = next(i for i, line in enumerate(lines[start + 1 :], start=start + 1) if line.startswith("2.5.12."))
fragment = lines[start:end]
image_links = [line for line in fragment if "![" in line or "](image_" in line]
print("Фрагмент после правок: data/extracted/2.5.md, пункт 2.5.11")
print("В сырой конвертации здесь были повторяющиеся ссылки на рисунки.")
print(f"Ссылок на изображения в пункте сейчас: {len(image_links)}")
print()
for number, line in enumerate(fragment, start=start + 1):
    print(f"{number:4}| {line}")


Фрагмент после правок: data/extracted/2.5.md, пункт 2.5.11
В сырой конвертации здесь были повторяющиеся ссылки на рисунки.
Ссылок на изображения в пункте сейчас: 0

 119| 2.5.11. Основными характеристиками нагрузок являются их нормативные значения, которые устанавливаются настоящими Правилами, а для нагрузок, не регламентированных ими, - в соответствии со строительными нормами и правилами.
 120| 
 121| Расчетные значения нагрузок определяются как произведение их нормативных значений на коэффициенты надежности по нагрузке $\gamma_f$, надежности по ответственности $\gamma_n$, условий работы $\gamma_d$, региональные $\gamma_p$.
 122| 
 123| При расчете элементов ВЛ расчетные нагрузки могут дополнительно умножаться на коэффициент сочетаний.
 124| 
 125| Необходимость применения коэффициентов и их значения устанавливаются настоящими Правилами.
 126| 
 127| При отсутствии указаний о значениях коэффициентов они принимаются равными единице.
 128| 


## 3. Чанкирование файлов extracted<a id="sec-3"></a>


Отредактированные файлы Markdown (extracted) преобразованы в формат JSON Lines (JSONL). Для последующей индексации выполнено разбиение на семантические чанки для последующей индексации в векторной базе данных.

**Цель чанкирования:** Подготовка структурированных фрагментов документации для эффективного векторного поиска — каждый чанк содержит логически завершённый пункт правил с полным контекстом (Раздел → Глава → Пункт).

**Основные результаты:**
- Файлы `*.chunked.jsonl` сохранены в директории `data/chunked`
- Каждая строка JSONL — отдельный чанк с полями `id`, `metadata`, `content`
- Сохранена иерархия документа: Document, Section, Chapter, Paragraph, Clause
- Изображения и таблицы остаются внутри чанков (флаги `contains_images`, `contains_tables`)

**Стратегия чанкирования:**
- Разбиение по пунктам по шаблону (X.Y.Z) — один пункт = один чанк
- Парсинг структуры MD: заголовки ###, ##, #, нумерованные пункты (1.2.3)
- Изображения из папок `image_*` копируются в выходную директорию

Подробное описание каждого подэтапа представлено в следующих разделах.


### 3.1. Результат (chunked)


После выполнения чанкирования получены следующие результаты:

**Расположение выходных данных:**

| Путь | Содержимое |
|------|------------|
| [data/chunked](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/data/chunked) | Файлы `*.chunked.jsonl` по главам ПУЭ |
| `data/chunked/image_*` | Папки с изображениями, скопированными из extracted |

**Структура выходных файлов:**

- Формат: JSON Lines (JSONL) — каждая строка файла — отдельный JSON-объект
- Именование: `{номер_главы}.chunked.jsonl` (например: `1.1.chunked.jsonl`, `4.4.chunked.jsonl`, `6.5.chunked.jsonl`)
- Соответствие: один MD-файл из `extracted` → один JSONL-файл в `chunked`

**Характеристики чанков:**

- Каждый чанк — логически завершённый пункт правил (X.Y.Z)
- Сохранён полный контекст: Document, Section, Chapter, Paragraph, Clause
- Изображения и таблицы встроены в контент 


Структура директории результатов чанкирования:


In [6]:
import datetime

chunked = ROOT / "data" / "chunked"
print("Команда: Get-ChildItem data/chunked")
print(f"Каталог: {chunked}")
print(f"{'Режим':<8} {'Дата':<18} {'Размер':>10}  Имя")
entries = sorted(chunked.iterdir(), key=lambda item: (item.is_file(), item.name.lower()))
for item in entries:
    mode = "d-----" if item.is_dir() else "a-----"
    stamp = datetime.datetime.fromtimestamp(item.stat().st_mtime).strftime("%d.%m.%Y %H:%M")
    size = "" if item.is_dir() else str(item.stat().st_size)
    print(f"{mode:<8} {stamp:<18} {size:>10}  {item.name}")
print()
print(f"Каталогов: {sum(item.is_dir() for item in entries)}")
print(f"Файлов: {sum(item.is_file() for item in entries)}")


Команда: Get-ChildItem data/chunked
Каталог: X:\Учеба_УИИ\Итоговы_Проект\data\chunked
Режим    Дата                   Размер  Имя
d-----   24.02.2026 15:55               image_1.3
d-----   24.02.2026 15:44               image_1.7
d-----   24.02.2026 15:55               image_1.9
d-----   24.02.2026 15:55               image_2.1
d-----   24.02.2026 15:55               image_2.3
d-----   24.02.2026 15:55               image_2.4
d-----   24.02.2026 15:55               image_2.5
d-----   24.02.2026 15:55               image_3.2
d-----   24.02.2026 15:55               image_3.4
d-----   24.02.2026 15:55               image_4.2
d-----   24.02.2026 15:55               image_4.3
d-----   24.02.2026 15:55               image_4.4
d-----   24.02.2026 15:55               image_5.4
d-----   24.02.2026 15:55               image_5.5
d-----   24.02.2026 15:55               image_6.1
d-----   24.02.2026 15:55               image_6.3
d-----   24.02.2026 15:55               image_6.6
d-----   24.02.2026 

### 3.2. Пример чанка (структура)


Каждая строка JSONL-файла представляет собой JSON-объект со следующей структурой:

**Основные поля чанка:**

| Поле | Описание |
|------|----------|
| `id` | Уникальный идентификатор (например, `pue_1.1.39_9ab608fe`) |
| `metadata` | Метаданные: Document, Section, Chapter, Paragraph, Clause, source_file, contains_tables, contains_images |
| `content` | Текст пункта с полным контекстом: Document. Section. Paragraph. Пункт X.Y.Z, затем сам текст пункта |
| `created_at` | Временная метка создания чанка |

**Структура metadata:**

- `Document` — название документа (например, «ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК»)
- `Section` — раздел (например, «Раздел 1 ОБЩИЕ ПРАВИЛА»)
- `Chapter` — глава (например, «Глава 1.7 - …»; может быть пустым для пунктов в «Общих указаниях»)
- `Paragraph` — параграф/подраздел (например, «Общие указания по устройству электроустановок»)
- `Clause` — номер пункта (например, «1.1.39»)
- `source_file` — исходный MD-файл
- `contains_tables` — наличие таблиц в чанке
- `contains_images` — наличие изображений в чанке

**Пример чанка (пункт 1.1.39):**

```json
{
    "id": "pue_1.1.39_9ab608fe",
    "metadata": {
        "Document": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК",
        "Section": "Раздел 1 ОБЩИЕ ПРАВИЛА",
        "Chapter": "",
        "Paragraph": "Общие указания по устройству электроустановок",
        "Clause": "1.1.39",
        "contains_tables": false,
        "contains_images": false,
        "source_file": "1.1.md"
    },
    "content": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК. Раздел 1 ОБЩИЕ ПРАВИЛА. Общие указания по устройству электроустановок. Пункт 1.1.39.\n\n1.1.39. Вновь сооруженные и реконструированные электроустановки вводятся в промышленную эксплуатацию только после их приемки согласно действующим положениям.",
    "created_at": "2026-03-06T00:42:08.846545"
}
```

Следующие ячейки печатают чанк пункта 1.1.39: сначала исходная строка JSONL, затем тот же объект через `json.dumps(..., indent=4)`.


In [7]:
import json

path = ROOT / "data" / "chunked" / "1.1.chunked.jsonl"
found = None
with path.open(encoding="utf-8") as handle:
    for line in handle:
        obj = json.loads(line)
        if obj.get("metadata", {}).get("Clause") == "1.1.39":
            found = line.rstrip("\n")
            break
print(f"Файл: {path}")
print("Исходная строка JSONL, пункт 1.1.39")
print()
print(found if found else "Чанк 1.1.39 не найден")


Файл: X:\Учеба_УИИ\Итоговы_Проект\data\chunked\1.1.chunked.jsonl
Исходная строка JSONL, пункт 1.1.39

{"id": "pue_1.1.39_9ab608fe", "metadata": {"Document": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК", "Section": "Раздел 1 ОБЩИЕ ПРАВИЛА", "Chapter": "", "Paragraph": "Общие указания по устройству электроустановок", "Clause": "1.1.39", "contains_tables": false, "contains_images": false, "source_file": "1.1.md"}, "content": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК. Раздел 1 ОБЩИЕ ПРАВИЛА. Общие указания по устройству электроустановок. Пункт 1.1.39.\n\n1.1.39. Вновь сооруженные и реконструированные электроустановки вводятся в промышленную эксплуатацию только после их приемки согласно действующим положениям.\n\nТекст документа сверен по:\n\nнормативно-производственное издание\n\nМ.: Издательство НЦ ЭНАС, 2002", "created_at": "2026-03-06T00:42:08.846545"}


In [8]:
import json

path = ROOT / "data" / "chunked" / "1.1.chunked.jsonl"
found = None
with path.open(encoding="utf-8") as handle:
    for line in handle:
        obj = json.loads(line)
        if obj.get("metadata", {}).get("Clause") == "1.1.39":
            found = obj
            break
print(f"Файл: {path}")
print("json.dumps(..., indent=4), пункт 1.1.39")
print()
if found is None:
    print("Чанк 1.1.39 не найден")
else:
    print(json.dumps(found, ensure_ascii=False, indent=4))


Файл: X:\Учеба_УИИ\Итоговы_Проект\data\chunked\1.1.chunked.jsonl
json.dumps(..., indent=4), пункт 1.1.39

{
    "id": "pue_1.1.39_9ab608fe",
    "metadata": {
        "Document": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК",
        "Section": "Раздел 1 ОБЩИЕ ПРАВИЛА",
        "Chapter": "",
        "Paragraph": "Общие указания по устройству электроустановок",
        "Clause": "1.1.39",
        "contains_tables": false,
        "contains_images": false,
        "source_file": "1.1.md"
    },
    "content": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК. Раздел 1 ОБЩИЕ ПРАВИЛА. Общие указания по устройству электроустановок. Пункт 1.1.39.\n\n1.1.39. Вновь сооруженные и реконструированные электроустановки вводятся в промышленную эксплуатацию только после их приемки согласно действующим положениям.\n\nТекст документа сверен по:\n\nнормативно-производственное издание\n\nМ.: Издательство НЦ ЭНАС, 2002",
    "created_at": "2026-03-06T00:42:08.846545"
}


### 3.3. Используемые скрипты


Директория: [src/preprocessing/Create_chunkeds](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/src/preprocessing/Create_chunkeds)

Для автоматизации процесса чанкирования были разработаны специализированные Python-скрипты:

| Скрипт | Назначение | Основные функции |
|--------|------------|------------------|
| [md_to_chunked_2.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_chunkeds/md_to_chunked_2.py) | Обработка одного файла | Парсинг структуры MD (PueMetadataParser), разбиение на чанки по пунктам, копирование изображений, запись JSONL |
| [generator.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_chunkeds/generator.py) | Массовая обработка | Асинхронная пакетная обработка всех MD-файлов из `data/extracted`, логирование процесса |

**Процесс работы скриптов:**

1. **md_to_chunked_2.py** — чтение MD-файла, парсинг заголовков и пунктов (###, ##, #, X.Y.Z), формирование чанков с метаданными, копирование изображений из папок `image_*` в выходную директорию, запись в `{глава}.chunked.jsonl`
2. **generator.py** — поиск всех `*.md` в `data/extracted`, параллельная конвертация (по умолчанию 4 потока), вывод лога по каждому файлу

Следующая ячейка печатает отчёт по `data/chunked`: размер и число чанков в каждом JSONL, отдельно строка для `2.5.md`.


In [9]:
chunked = ROOT / "data" / "chunked"
rows = []
for path in sorted(chunked.glob("*.jsonl")):
    with path.open(encoding="utf-8") as handle:
        count = sum(1 for line in handle if line.strip())
    rows.append((path.name, path.stat().st_size, count))

print("Отчёт по уже созданным JSONL (повторная нарезка не запускается).")
print("Команда нарезки одного файла:")
print("python src/preprocessing/Create_chunkeds/md_to_chunked_2.py data/extracted/2.5.md data/chunked")
print()
print(f"{'Файл':<32} {'Байт':>10} {'Чанков':>8}")
for name, size, count in rows:
    print(f"{name:<32} {size:10} {count:8}")
print()
total = sum(count for _, _, count in rows)
print(f"Файлов: {len(rows)}")
print(f"Чанков всего: {total}")
for name, size, count in rows:
    if name.startswith("2.5"):
        print()
        print(f"📂 Обработка файла '2.5.md'")
        print(f"🔍 Найдено чанков в '2.5.md': {count}")
        print(f"✅ Успешно создано {count} чанков из '2.5.md'.")
        print(f"💾 Результат сохранен: {chunked / name} ({size} байт)")


Отчёт по уже созданным JSONL (повторная нарезка не запускается).
Команда нарезки одного файла:
python src/preprocessing/Create_chunkeds/md_to_chunked_2.py data/extracted/2.5.md data/chunked

Файл                                   Байт   Чанков
1.1.chunked.jsonl                     58563       40
1.2.chunked.jsonl                     37839       25
1.3.chunked.jsonl                    105120       33
1.4.chunked.jsonl                     39752       21
1.5.chunked.jsonl                     69219       44
1.6.chunked.jsonl                     39352       23
1.7.chunked.jsonl                    318756      178
1.8.chunked.jsonl                    275897       42
1.9.chunked.jsonl                     89494       55
2.1.chunked.jsonl                    105986       80
2.2.chunked.jsonl                     44032       40
2.3.chunked.jsonl                    223426      148
2.4.chunked.jsonl                    124812       97
2.5.chunked.jsonl                    592677      296
3.1.chunked.js

## 4. Векторная база данных<a id="sec-4"></a>


Развёрнута векторная база данных Milvus для хранения эмбеддингов чанков нормативной документации ПУЭ. Milvus выбрана как система для хранения и поиска векторных представлени.

**Цель шага:** Обеспечение инфраструктуры для семантического поиска — хранение векторных представлений текста (и в перспективе изображений через CLIP) с возможностью быстрого поиска по сходству.

**Основные результаты:**
- Развёрнута векторная база данных Milvus (standalone) через docker-compose
- Настроен веб-интерфейс Attu для просмотра коллекций и данных
- Коллекция `diplom_multimodal` создаётся при загрузке чанков

**Инфраструктура:**

| Компонент | Описание |
|-----------|----------|
| infra/milvus | docker-compose: Milvus, etcd, MinIO (на основе [milvus-standalone-docker-compose](https://github.com/milvus-io/milvus/releases) https://github.com/milvus-io/milvus/releases|
| Attu | Веб-интерфейс для просмотра коллекций, схемы и записей (localhost:3000). (Docker: [zilliz/attu](https://hub.docker.com/r/zilliz/attu) https://hub.docker.com/r/zilliz/attu)|
| Порт Milvus | 19530 (gRPC) |

**Процедура развёртывания:**
1. Запуск контейнеров Milvus: `docker-compose up -d`
2. Запуск Attu: `docker run -d --name milvus-attu -p 3000:3000 -e MILVUS_URL=http://milvus-standalone:19530 --network milvus zilliz/attu:v2.4.0`
3. Проверка статуса: `docker-compose ps`, `docker-compose logs -f milvus-standalone`
4. Доступ к Attu по адресу localhost:3000 — просмотр коллекций и данных (после загрузки чанков в разделе 5)

Подробное описание представлено в следующих подразделах.


### 4.1. Результат развёртывания (Attu)


После запуска Milvus и загрузки чанков (раздел 5) коллекция доступна в Attu (localhost:3000). Следующие ячейки печатают те же сведения: доступность сервера, список коллекций, число сущностей, примеры записей и схема с индексами HNSW.


In [10]:
print("Отчёт вместо экрана подключения Attu")
info = connect_milvus()
if info is not None:
    print(f"Attu: http://localhost:3000")
    print(f"Milvus gRPC: доступен, коллекция для просмотра — {info['name']}")


Отчёт вместо экрана подключения Attu
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


In [11]:
print("Отчёт вместо главного экрана Attu")
info = connect_milvus()
if info is None:
    pass
elif not info["utility"].has_collection(info["name"], using=info["alias"]):
    print(f"Коллекция {info['name']} не найдена.")
    print("Коллекции:", info["utility"].list_collections(using=info["alias"]))
else:
    names = info["utility"].list_collections(using=info["alias"])
    print("База: default")
    print("Коллекции:")
    for name in names:
        print(f"  - {name}")


Отчёт вместо главного экрана Attu
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


In [12]:
print("Отчёт по коллекции diplom_multimodal")
info = connect_milvus()
if info is None:
    pass
elif not info["utility"].has_collection(info["name"], using=info["alias"]):
    print(f"Коллекция {info['name']} не найдена.")
else:
    collection = info["Collection"](info["name"], using=info["alias"])
    print(f"Имя: {collection.name}")
    print(f"Описание: {collection.description}")
    print(f"Сущностей: {collection.num_entities}")
    print(f"Поля: {[field.name for field in collection.schema.fields]}")


Отчёт по коллекции diplom_multimodal
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


In [13]:
print("Примеры записей коллекции (текст и метаданные, без полного вектора)")
info = connect_milvus()
if info is None:
    pass
elif not info["utility"].has_collection(info["name"], using=info["alias"]):
    print(f"Коллекция {info['name']} не найдена.")
else:
    collection = info["Collection"](info["name"], using=info["alias"])
    collection.load()
    rows = collection.query(
        expr='chunk_id != ""',
        output_fields=["chunk_id", "chapter", "source_file", "has_image", "text"],
        limit=3,
    )
    print(f"Показано записей: {len(rows)}")
    for index, row in enumerate(rows, start=1):
        text = (row.get("text") or "").replace("\n", " ")
        if len(text) > 280:
            text = text[:280] + "..."
        print()
        print(f"{index}. chunk_id: {row.get('chunk_id')}")
        print(f"   chapter: {row.get('chapter')}")
        print(f"   source_file: {row.get('source_file')}")
        print(f"   has_image: {row.get('has_image')}")
        print(f"   text: {text}")


Примеры записей коллекции (текст и метаданные, без полного вектора)
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


In [14]:
print("Схема коллекции и параметры индексов HNSW")
info = connect_milvus()
if info is None:
    pass
elif not info["utility"].has_collection(info["name"], using=info["alias"]):
    print(f"Коллекция {info['name']} не найдена.")
else:
    collection = info["Collection"](info["name"], using=info["alias"])
    print(f"Коллекция: {collection.name}")
    for field in collection.schema.fields:
        extra = ""
        params = getattr(field, "params", None) or {}
        if params:
            extra = " " + ", ".join(f"{key}={value}" for key, value in params.items())
        print(f"  {field.name}: {field.dtype}{extra}")
    print("Индексы:")
    for index in collection.indexes:
        print(f"  поле: {index.field_name}")
        print(f"  параметры: {index.params}")


Схема коллекции и параметры индексов HNSW
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


**Схема коллекции:**

| Поле | Описание |
|------|----------|
| `id` | Уникальный идентификатор записи |
| `chunk_id` | Идентификатор чанка (например, pue_1.1.1_xxx) |
| `text_vector` | Векторное представление текста (модель intfloat/multilingual-e5-base) |
| `image_vector` | Векторное представление изображения (CLIP) |
| `text` | Текст чанка |
| `image_paths` | Пути к изображениям в чанке |
| `source_file` | Исходный MD-файл |
| `chapter` | Глава ПУЭ |
| `has_image` | Наличие изображений в чанке |


## 5. Загрузка в векторную базу данных и тестовые запросы<a id="sec-5"></a>


Загружены чанки из `data/chunked` в векторную базу данных Milvus. Работа семантического поиска проверена с помощью тестовых запросов.

**Цель шага:** Индексация подготовленных чанков в векторной базе данных для быстрого поиска релевантных фрагментов по пользовательскому запросу и валидация качества поиска.

**Основные результаты:**
- Чанки загружены в коллекцию `diplom_multimodal`.
- Эмбеддинги текста вычисляются при загрузке (модель intfloat/multilingual-e5-base).
- Поддержка изображений заложена в структуру данных (планируется векторизация через CLIP).
- Тестовые запросы подтверждают релевантность результатов поиска.

**Процесс загрузки:**
1. Чтение JSONL-файлов из `data/chunked`
2. Вычисление эмбеддингов текста для каждого чанка
3. Запись в Milvus (batch-загрузка)
4. Тестовые запросы через `query_test.py`

Подробное описание скриптов и примеров запросов представлено в следующих подразделах.


### 5.1. Используемые скрипты


Директория: [src/preprocessing/Create_embeddings](https://github.com/kirag-ozyaz/Diplom_AI/tree/main/src/preprocessing/Create_embeddings)

Для загрузки данных и тестирования поиска используются следующие Python-скрипты:

| Скрипт | Назначение | Основные функции |
|--------|------------|------------------|
| [load_data.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_embeddings/load_data.py) | Загрузка данных в коллекцию | Создание коллекции, чтение JSONL из `data/chunked`, вычисление эмбеддингов, batch-вставка в Milvus, вывод статистики |
| [multimodal_rag.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_embeddings/multimodal_rag.py) | Ядро RAG-системы | Класс MultimodalRAG: создание коллекции, загрузка чанков, поиск по тексту (и изображениям), поддержка асинхронной загрузки |
| [query_test.py](https://github.com/kirag-ozyaz/Diplom_AI/blob/main/src/preprocessing/Create_embeddings/query_test.py) | Тестовые запросы | Подключение к готовой коллекции, семантический поиск по запросу, вывод топ-N результатов с score, главой и текстом |

**Процесс работы:**

1. **load_data.py** — создаёт коллекцию (при необходимости с `drop_existing=True`), загружает все JSONL из `data/chunked`, вычисляет эмбеддинги, выводит статистику (количество сущностей, поля схемы)
2. **query_test.py** — подключается к уже загруженной коллекции, выполняет тестовый запрос (например, «Нулевой защитный и нулевой рабочий проводники»), выводит результаты с score и фрагментами текста

Повторный запуск `load_data.py` из ноутбука не выполняется: в конфиге `drop_existing` равен true, и коллекция была бы пересоздана. Ячейки ниже печатают состав `data/chunked` и, если Milvus запущен, статистику уже загруженной коллекции.


In [15]:
print("Вход загрузки: data/chunked")
print("Команда загрузки (в этой ячейке не выполняется, drop_existing пересоздаёт коллекцию):")
print("python src/preprocessing/Create_embeddings/load_data.py")
print()
chunked = ROOT / "data" / "chunked"
total = 0
files = 0
for path in sorted(chunked.glob("*.jsonl")):
    with path.open(encoding="utf-8") as handle:
        count = sum(1 for line in handle if line.strip())
    total += count
    files += 1
    print(f"  {path.name}: {count} чанков")
print()
print(f"Файлов JSONL: {files}")
print(f"Чанков к загрузке: {total}")


Вход загрузки: data/chunked
Команда загрузки (в этой ячейке не выполняется, drop_existing пересоздаёт коллекцию):
python src/preprocessing/Create_embeddings/load_data.py

  1.1.chunked.jsonl: 40 чанков
  1.2.chunked.jsonl: 25 чанков
  1.3.chunked.jsonl: 33 чанков
  1.4.chunked.jsonl: 21 чанков
  1.5.chunked.jsonl: 44 чанков
  1.6.chunked.jsonl: 23 чанков
  1.7.chunked.jsonl: 178 чанков
  1.8.chunked.jsonl: 42 чанков
  1.9.chunked.jsonl: 55 чанков
  2.1.chunked.jsonl: 80 чанков
  2.2.chunked.jsonl: 40 чанков
  2.3.chunked.jsonl: 148 чанков
  2.4.chunked.jsonl: 97 чанков
  2.5.chunked.jsonl: 296 чанков
  3.1.chunked.jsonl: 20 чанков
  3.2.chunked.jsonl: 133 чанков
  3.3.chunked.jsonl: 103 чанков
  3.4.chunked.jsonl: 30 чанков
  4.1.chunked.jsonl: 31 чанков
  4.2.chunked.jsonl: 244 чанков
  4.3.chunked.jsonl: 53 чанков
  4.4.chunked.jsonl: 46 чанков
  5.1.chunked.jsonl: 38 чанков
  5.2.chunked.jsonl: 61 чанков
  5.3.chunked.jsonl: 63 чанков
  5.4.chunked.jsonl: 69 чанков
  5.5.chunked.jso

In [16]:
print("Статистика уже загруженной коллекции")
print("Соответствует финальному блоку load_data.py, без повторной загрузки.")
info = connect_milvus()
if info is None:
    pass
elif not info["utility"].has_collection(info["name"], using=info["alias"]):
    print(f"Коллекция {info['name']} не найдена. Сначала выполните load_data.py.")
else:
    collection = info["Collection"](info["name"], using=info["alias"])
    fields = [field.name for field in collection.schema.fields]
    print("Статистика коллекции:")
    print(f"   Название: {collection.name}")
    print(f"   Сущностей: {collection.num_entities}")
    print(f"   Поля: {fields}")
    description = collection.description or ""
    print(f"   Описание: {description}")


Статистика уже загруженной коллекции
Соответствует финальному блоку load_data.py, без повторной загрузки.
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


In [17]:
import re

print("Метаданные эмбеддингов и индексы")
info = connect_milvus()
if info is None:
    pass
elif not info["utility"].has_collection(info["name"], using=info["alias"]):
    print(f"Коллекция {info['name']} не найдена.")
else:
    collection = info["Collection"](info["name"], using=info["alias"])
    description = collection.description or ""
    match = re.search(r"text_model=([^,]+),\s*text_dim=(\d+)", description)
    if match:
        print(f"text_model={match.group(1).strip()}")
        print(f"text_dim={match.group(2)}")
    else:
        print("В описании коллекции нет text_model/text_dim.")
        print(description)
    for index in collection.indexes:
        print(f"index {index.field_name}: {index.params}")


Метаданные эмбеддингов и индексы
Сервер: localhost:19530
Коллекция: diplom_multimodal
Milvus недоступен. Запустите контейнеры из infra/milvus и повторите ячейку.


### 5.2. Результаты тестовых запросов


Тестовые запросы выполняются через `query_test.py` и демонстрируют работу семантического поиска по загруженной коллекции.

**Основные результаты:**

| Материал | Описание |
|----------|----------|
| Пример запроса | Текст запроса (например: «система заземления», «нулевой защитный проводник», «защитное заземление») |
| Топ-N результатов | Для каждого результата: score (релевантность), chapter (глава), фрагмент текста, наличие изображений |
| Сводная таблица | Запрос → Найденный пункт (1.1.X) → Score |

**Формат вывода query_test.py:**

- Проверка доступности сервера векторной базы данных
- Модель эмбеддингов из метаданных коллекции
- Для каждого результата: номер, score, глава, текст (или snippet), флаг наличия изображений


Следующая ячейка выполняет тот же поиск, что и `query_test.py`, и печатает score, главу и текст найденных пунктов. Нужны запущенный Milvus и модель эмбеддингов.


In [ ]:
import sys

print("Команда: python src/preprocessing/Create_embeddings/query_test.py")
print()
host, port, collection_name = milvus_settings()
if not milvus_reachable(host, port):
    print(f"Сервер векторной БД недоступен: {host}:{port}")
    print("Запустите Milvus и повторите ячейку.")
else:
    sys.path.insert(0, str(ROOT / "src" / "preprocessing" / "Create_embeddings"))
    from multimodal_rag import MultimodalRAG, get_default_embedding_model
    from runtime_config import load_runtime_config
    from model_selector import select_text_model

    cfg = load_runtime_config()
    query_test_cfg = cfg["query_test"]
    models = cfg["models"]
    print(f"Сервер векторной БД доступен: {host}:{port}")
    meta = MultimodalRAG.get_embedding_meta_from_collection(host, port, collection_name)
    if meta:
        text_model_name = meta["text_model"]
        text_dim = meta["text_dim"]
        print(f"Модель из метаданных коллекции: {text_model_name}, text_dim={text_dim}")
    else:
        text_model_name, reason = select_text_model(cfg)
        text_dim = None
        if not text_model_name:
            text_model_name, text_dim = get_default_embedding_model()
            print(f"Модель по умолчанию: {text_model_name}, text_dim={text_dim}")
        else:
            print(f"Автовыбор модели: {text_model_name} ({reason})")

    rag = MultimodalRAG(
        vector_db_host=host,
        vector_db_port=port,
        collection_name=collection_name,
        text_model_name=text_model_name,
        text_dim=text_dim,
        base_data_path=str(ROOT / "data"),
        device_text=models["device_text"],
        device_clip=models["device_clip"],
        load_image_model=False,
    )
    rag.load_collection()
    search_text = str(query_test_cfg["search_text"])
    test_limit = int(query_test_cfg["limit"])
    print()
    print("Тестовый поиск")
    print(f"Текст для поиска: {search_text}")
    results = rag.search_text(search_text, limit=test_limit)
    for index, res in enumerate(results, 1):
        print()
        print(f"{index}. Score: {res['score']:.4f}")
        print(f"   Глава: {res['chapter']}")
        text = res.get("text") or ""
        print(f"   Текст: {text if text.strip() else '(пусто)'}")
        if res.get("has_image"):
            print(f"   Изображений: {len(res.get('image_paths') or [])}")
    rag.close()
    print()
    print("Тест завершён")
